In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# define the catalog and database
catalog = 'uk_property_insights'
database = 'default'
bronze_table_name = 'bronze_property_prices'
silver_table_name = "silver_property_prices"
silver_rejects_table_name = "silver_property_prices_rejects"

# set the catalog and database
spark.catalog.setCurrentCatalog(catalog)
spark.catalog.setCurrentDatabase(database)
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current database: {spark.catalog.currentDatabase()}")

# check if the tables exist
bronze_table_exists = spark.catalog.tableExists(bronze_table_name)
silver_table_exists = spark.catalog.tableExists(silver_table_name)
silver_rejects_table_exists = spark.catalog.tableExists(silver_rejects_table_name)
print(f"Bronze table exists: {bronze_table_exists}")
print(f"Silver table exists: {silver_table_exists}")

In [0]:
# drop the tables if they exist
if bronze_table_exists:
    query = f"DROP TABLE IF EXISTS {bronze_table_name}"
    spark.sql(query)
if silver_table_exists:
    query = f"DROP TABLE IF EXISTS {silver_table_name}"
    spark.sql(query)
if silver_rejects_table_exists:
    query = f"DROP TABLE IF EXISTS {silver_rejects_table_name}"
    spark.sql(query)

In [0]:
# define the bronze table schema
bronze_schema = StructType([
    # Raw source data
    StructField("transaction_id", StringType(), True),
    StructField("price", StringType(), True),
    StructField("transfer_date", StringType(), True),
    StructField("postcode", StringType(), True),
    StructField("property_type", StringType(), True),
    StructField("new_build", StringType(), True),
    StructField("tenure", StringType(), True),
    StructField("primary_addressable_object_name", StringType(), True),
    StructField("secondary_addressable_object_name", StringType(), True),
    StructField("street", StringType(), True),
    StructField("locality", StringType(), True),
    StructField("town_city", StringType(), True),
    StructField("district", StringType(), True),
    StructField("county", StringType(), True),
    StructField("ppd_category_type", StringType(), True),
    StructField("record_status", StringType(), True),
    # Metadata
    StructField("ingestion_time", TimestampType(), False),
    StructField("source_file", StringType(), False)
])

# create the bronze table
empty_df = spark.createDataFrame([], bronze_schema)

empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(bronze_table_name)



In [0]:
# define the silver table schema
silver_schema = StructType([
    # Required columns
    StructField("transaction_id", StringType(), False),
    StructField("price", IntegerType(), False),
    StructField("transfer_date", TimestampType(), False),
    StructField("postcode", StringType(), False),
    # Optional columns
    StructField("property_type", StringType(), True),
    StructField("new_build", StringType(), True),
    StructField("tenure", StringType(), True),
    StructField("primary_addressable_object_name", StringType(), True),
    StructField("secondary_addressable_object_name", StringType(), True),
    StructField("street", StringType(), True),
    StructField("locality", StringType(), True),
    StructField("town_city", StringType(), True),
    StructField("district", StringType(), True),
    StructField("county", StringType(), True),
    StructField("ppd_category_type", StringType(), True),
    StructField("record_status", StringType(), True),
    # Metadata
    StructField("ingestion_time", TimestampType(), False),
    StructField("source_file", StringType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

empty_df = spark.createDataFrame([], silver_schema)

empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table_name)

In [0]:
# define the silver rejects table schema
rejects_schema = StructType([
    # Raw source data
    StructField("transaction_id", StringType(), True),
    StructField("price", StringType(), True),
    StructField("transfer_date", StringType(), True),
    StructField("postcode", StringType(), True),
    StructField("property_type", StringType(), True),
    StructField("new_build", StringType(), True),
    StructField("tenure", StringType(), True),
    StructField("primary_addressable_object_name", StringType(), True),
    StructField("secondary_addressable_object_name", StringType(), True),
    StructField("street", StringType(), True),
    StructField("locality", StringType(), True),
    StructField("town_city", StringType(), True),
    StructField("district", StringType(), True),
    StructField("county", StringType(), True),
    StructField("ppd_category_type", StringType(), True),
    StructField("record_status", StringType(), True),
    # Metadata
    StructField("ingestion_time", TimestampType(), False),
    StructField("source_file", StringType(), False),
    StructField("rejection_reason", StringType(), False)
])

# create the bronze table
empty_df = spark.createDataFrame([], rejects_schema)

empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_rejects_table_name)
